## 0. Setup

In [ ]:
# If running outside the repo root, uncomment to pip-install:
# !pip install -e "..[viz]"

import os
import sys
import tempfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Crisper inline figures in the notebook.
%config InlineBackend.figure_format = 'retina'
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 200
plt.rcParams['image.interpolation'] = 'nearest'

# When running from the repo, make sure the local package is importable.
sys.path.insert(0, str(Path.cwd().parent))

import frameselect as fs
from frameselect import Selector

# Use the GPU when CUDA is available, otherwise the CPU.
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"frameselect loaded — {len(Selector.list())} registered selectors")
print(f"device           : {DEVICE}")

## 1. Point the selector at a video

Set `VIDEO` to the path of a video file of your own. `.select()` also accepts a remote URL (`http`, `https`, `ftp`, or `s3` / `gs` with the `cloud` extra), which the library downloads to a temporary file on demand, or the frames themselves as an array.

In [ ]:
VIDEO = "path/to/video.mp4"  # any local video file
assert VIDEO.startswith(("http://", "https://", "ftp://", "s3://", "gs://")) or Path(VIDEO).exists(), (
    f"Set VIDEO to a video file on this machine (or a URL); {VIDEO!r} does not exist."
)

# One-liner: pick 6 uniform frames.
sel = Selector("uniform", n_frames=6)
result = sel.select(VIDEO)

print(f"Selected indices: {result.indices.tolist()}")
print(f"Selected frames : {result.frames.shape}  dtype={result.frames.dtype}")
print(f"Score kind      : {result.score_kind}")
video_keys = ("source", "total_frames", "native_fps", "duration_sec")
print(f"Video metadata  : { {k: result.metadata.get(k) for k in video_keys} }")

## 2. Compare six selectors on the same video

Six different strategies, from a fixed baseline to query-aware selection. Each is a different way to pick frames:

| Selector    | Strategy |
|---          |---|
| `uniform`   | Fixed temporal spacing -- the baseline |
| `kmeans`    | K-Means on CLIP embeddings -- one frame per visual theme (query-agnostic) |
| `efs`       | Event-anchored: cuts the video into visual events, anchors each event on its most query-relevant frame, then adds relevant frames that are not redundant with those already picked |
| `agfs`      | Adaptive greedy: adds frames one at a time by their gain in query relevance plus coverage of the whole candidate pool |
| `qav`       | Query-answer alignment: keeps the frames that best match the query combined with any one of its answer choices |
| `medr`      | Multi-signal events: visual-change and optical-flow events computed from pixels (no encoder, no query), then greedy rescoring that weighs each frame's signal score and event coverage against its temporal redundancy |

This side-by-side only runs here because the library wraps every method behind one `Selector(name).select(video, query)` call. Each row below is one such call; they all share the video loader, the 1 fps candidate pool (capped at 128 frames) and a uniform `SelectionResult` contract, and the four embedding-based selectors (`kmeans`, `efs`, `agfs`, `qav`) share the CLIP ViT-B-32 encoder -- so the comparison is protocol-matched, not a hand-curated table. The loop passes the query only to the query-aware selectors (`efs`, `agfs`, `qav`). `qav` also reads multiple-choice answer choices (it sets `uses_answer_choices`), so it gets two of them after the query, one per line -- the format the evaluators use to pass a benchmark's options. Given the plain query alone, it would keep the frames most similar to the query, the QAV paper's question-only variant.

In [ ]:
# Edit QUERY and CHOICES to describe your video.
QUERY = "a person holding an object"
# `qav` also reads multiple-choice answer choices, one per line after the
# query (the format evaluators use for a benchmark's options). With two or
# more, it scores each frame by its best match over the "query + choice" texts.
CHOICES = ["A. indoors", "B. outdoors"]
N_FRAMES = 8

CACHE_DIR = Path(tempfile.gettempdir()) / "frameselect_cache"
CACHE_DIR.mkdir(exist_ok=True)

def make(name, **extra):
    return Selector(
        name,
        n_frames=N_FRAMES,
        cache_dir=str(CACHE_DIR),
        device=DEVICE,
        **extra,
    )

# Default encoder: CLIP ViT-B-32 (~600 MB of weights). Light enough to run on
# a laptop without a GPU. To match the paper's shared encoder, swap to
# `encoder="ViT-SO400M-16-SigLIP2-384", encoder_pretrained="webli"`
# -- that pulls ~4.6 GB on first call and needs a GPU.
# `uniform` and `medr` load no encoder; `medr` computes its signals with OpenCV.
selectors = {
    "uniform": make("uniform"),
    "kmeans":  make("kmeans"),
    "efs":     make("efs"),
    "agfs":    make("agfs"),
    "qav":     make("qav"),
    "medr":    make("medr"),
}

results = {}
for name, sel in selectors.items():
    if not sel.query_aware:
        kw = {}
    elif sel.uses_answer_choices:  # qav: the query, then its answer choices
        kw = dict(query="\n".join([QUERY, *CHOICES]))
    else:
        kw = dict(query=QUERY)
    try:
        r = sel.select(VIDEO, **kw)
        results[name] = r
        print(f"{name:12s} indices={r.indices.tolist()}")
    except Exception as e:
        print(f"{name:12s} SKIPPED ({type(e).__name__}: {str(e)[:80]})")

# Grid of picks -- one row per selector, no time axis
fs.viz.selection_strip(results, thumb_width=180)

## 3. Selection across different queries

Run the same query-aware selector, `efs`, with different queries to see how query-awareness actually behaves. For a local video file the embedding cache is keyed on `(video, encoder, fps)`, so repeated queries skip the encoder pass entirely -- the cost of a new query is one text-encoder call and one selection step.

`fs.viz.selection_strip` lays out one row per (method or query) with thumbnails of the selected frames.

In [ ]:
sel = Selector("efs", n_frames=6, cache_dir=str(CACHE_DIR), device=DEVICE)

queries = [  # edit these to describe your video
    "a close-up of a face",
    "text or a logo on screen",
    "a hand holding an object",
    "a wide shot of the whole scene",
]
per_query = {q: sel.select(VIDEO, query=q) for q in queries}

# lays thumbnails out in a regular grid — no overlap,
# every pick fully visible. Best when you want to inspect the frames
# themselves rather than their timestamps.
fs.viz.selection_strip(per_query, thumb_width=220)

Each row is the *same selector* answering a different query about the same video. EFS cuts the video into the same visual events for every query (at most six at this 6-frame budget) and anchors each event on its most query-relevant frame, so the query decides *which* frame of each event is picked. Compare the rows to see the picks move toward the frames that match each query. This is the core behaviour query-aware selectors provide. A query-independent selector -- `medr`, like `uniform` and `kmeans` -- would return the same row four times, which is why one of its selections can serve every question about a video.

## 4. Zero-setup programmatic access

`result.frames` is a plain NumPy array — hand it to any downstream system (VLM, captioner, classifier, retrieval, etc.).

In [ ]:
r = Selector("efs", n_frames=4, cache_dir=str(CACHE_DIR), device=DEVICE).select(
    VIDEO, query=QUERY
)

# The frames themselves are a plain (N, H, W, 3) uint8 array:
print("frames :", r.frames.shape, r.frames.dtype)
print("indices:", r.indices.tolist())
print("scores :", r.scores)

# For a quick visual check, the result itself renders inline
# via ``selection_strip``:
fs.viz.selection_strip({"efs": r}, thumb_width=360)

## 5. Debug traces: *why* did a method pick these frames?

`fs.viz.debug_trajectory` is one call that renders whatever diagnostic signals a selector chooses to publish. The library side is small: a selector opts in by assigning a dict to `self._pending_debug_traces` while it selects (`select()` moves it to `result.metadata["debug_traces"]`), where each entry is `{values, label, kind}`, and selectors that don't opt in fall back to the generic score-timeline view automatically -- no error, no wrapper code, no per-method switch in the caller.

That extension hook is the point. The same `debug_trajectory(result)` call works for every selector, including ones a user registers themselves later. Two of the bundled selectors publish traces: `efs` (its temporal-similarity curve and its query relevance) and `medr` (its visual-stability and motion signals, plus a text signal when OCR is enabled). `uniform`, `kmeans`, `agfs` and `qav` publish none, so for them the same call draws the score timeline instead. The two cells below show EFS on the query from section 2, then MEDR, which takes no query.

In [ ]:
# EFS publishes two traces: the temporal-similarity curve its events are cut
# from, and the query relevance that picks each event's anchor frame.
r_efs = Selector("efs", n_frames=8, cache_dir=str(CACHE_DIR), device=DEVICE).select(
    VIDEO, query=QUERY
)
fs.viz.debug_trajectory(r_efs, title=f"EFS — query: {QUERY!r}");

In [ ]:
# MEDR takes no query: its traces are the pixel signals its events are built
# from -- visual stability and optical-flow motion.
r_medr = Selector("medr", n_frames=8).select(VIDEO)
fs.viz.debug_trajectory(r_medr, title="MEDR — no query");

What to read off, at the library level:

- Both plots above come from the same `fs.viz.debug_trajectory(result)` call, with no method-specific caller code: one stacked panel per published trace. A selector that publishes none, such as `agfs` or `qav`, falls through to `score_timeline(result)` -- still one line of caller code, still a useful plot.
- The red dots mark the selected frames, rendered uniformly from `result.indices`.
- EFS starts a new event at each dip (local minimum) of its temporal-similarity curve, then merges the most similar neighbouring events until no more than `max_events` remain, capped at the frame budget. When that leaves as many events as frames, each red dot on the relevance panel is the most query-relevant frame of its event.
- MEDR's visual-stability curve dips wherever consecutive frames change. Its picks come from the rescoring step, which weighs each frame's signal score and the event coverage it adds against its temporal redundancy with the frames already picked, so they need not sit on the peaks of either curve.
- A selector you register yourself gets the same view: assign `{values, label, kind}` entries to `self._pending_debug_traces` in its selection method and the panels appear without changing the caller.

## 6. List every registered selector

The library ships 6 selectors — this notebook used 6 of them. For references see the README, or list them:

In [ ]:
for name in Selector.list():
    cls = fs.registry.get_method(name)
    q = "✓" if getattr(cls, "query_aware", False) else "✗"
    print(f"  {name:15s}  query-aware: {q}  ({cls.__name__})")

## Next steps

The same `Selector(name).select(video, query)` call scales to a full benchmark run:

- `examples/run_activitynet.py` -- runs every registered selector on ActivityNet activity recognition under one shared configuration (SigLIP2-SO400M-384, 128 candidates, 16 output frames); each accuracy number comes from one `ev.run(sel)` call.

What the library adds over a single-method repo: one string-factory API for every selector, evaluators behind a matching `Evaluator(name)` factory, a shared frame-embedding cache, replay mode that swaps the answering VLM without re-selecting, per-instance VLM routing (no global state), and a uniform `SelectionResult` / `EvaluationResult` contract. Everything you saw here is one `Selector` / `Evaluator` call per row -- no copy-paste, no per-method harness code, no environment pin drift.